# Satellite Image Segmentation with Fuzzy C-Means

**Master's thesis experiment**: unsupervised segmentation of a satellite image with a from-scratch, vectorised
implementation of the **Fuzzy C-Means (FCM)** clustering algorithm, evaluated against a ground-truth mask.

Pipeline: `RGB image -> grayscale -> Gaussian blur -> normalisation -> Fuzzy C-Means -> hard labels (argmax) -> accuracy vs. ground truth`

| Parameter | Value |
|---|---|
| Fuzziness index `m` | 1.7 |
| Convergence threshold `epsilon` | 1e-5 |
| Maximum iterations | 300 |
| Number of clusters `k` | 6 in this notebook (the thesis also reports k = 2 ... 5, obtained by changing `k`) |
| Membership initialisation | random (Dirichlet), **no fixed seed**, so iteration counts vary slightly between runs |

See `docs/` for the theory, the methodology and the results.

## 0. Configuration
Place the image and its mask in `../data/` (see `data/README.md`).

In [ ]:
# Paths to the input image and its ground-truth mask (relative to this notebook)
IMAGE_PATH = "../data/606_sat1.jpg"
MASK_PATH = "../data/606_mask.png"

## 1. Load the image and the ground-truth mask

In [ ]:
import cv2
import matplotlib.pyplot as plt
import os
import numpy as np 
import pandas as pd 

# Nom des fichiers
image_path = IMAGE_PATH
mask_path = MASK_PATH

# Vérification de la présence des fichiers
if not os.path.exists(image_path):
    print(f"Erreur : Fichier image non trouvé -> {image_path}")
elif not os.path.exists(mask_path):
    print(f"Erreur : Fichier masque non trouvé -> {mask_path}")
else:
    # Chargement de l'image et du masque (en couleur BGR)
    image = cv2.imread(image_path)
    mask = cv2.imread(mask_path)

    # Conversion en RGB pour affichage correct
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    mask_rgb = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)

    # Affichage côte à côte des images en couleur
    plt.figure(figsize=(12, 6))

    plt.subplot(1, 2, 1)
    plt.imshow(image_rgb)
    plt.title("Image satellite (RGB)")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow(mask_rgb)
    plt.title("Masque (RGB)")
    plt.axis('off')

    plt.show()

## 2. Pre-processing: grayscale conversion and Gaussian blur (5x5)

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import os

# Chemins des fichiers
image_path = IMAGE_PATH
mask_path = MASK_PATH

# Vérification de la présence des fichiers
if not os.path.exists(image_path):
    print(f"Erreur : Fichier image non trouvé -> {image_path}")
elif not os.path.exists(mask_path):
    print(f"Erreur : Fichier masque non trouvé -> {mask_path}")
else:
    # Chargement de l'image et du masque
    image = cv2.imread(image_path)
    mask = cv2.imread(mask_path)

    # Conversion de l'image en niveaux de gris
    image_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # 1. Application du flou gaussien
    image_blur = cv2.GaussianBlur(image_gray, (5, 5), 0)

    # Affichage de l'image avant et après le flou gaussien
    plt.figure(figsize=(6, 6))

    # Affichage de l'image originale en niveaux de gris
    plt.subplot(1, 2, 1)
    plt.imshow(image_gray, cmap='gray')
    plt.title("Image originale en niveaux de gris")
    plt.axis('off')

    # Affichage après l'application du flou gaussien
    plt.subplot(1, 2, 2)
    plt.imshow(image_blur, cmap='gray')
    plt.title("Image après Flou Gaussien")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

    # Affichage des dimensions et valeurs après l'application du flou gaussien
    print("Prétraitement terminé avec Flou Gaussien.")
    print("Valeurs min/max après flou gaussien :", image_blur.min(), "/", image_blur.max())

Prétraitement terminé avec Flou Gaussien.
Valeurs min/max après flou gaussien : 24 / 254


## 3. Data preparation: normalisation to [0, 1] and flattening

In [ ]:
# --- Étape : Préparation des données pour FCM ---

# Normalisation : met les valeurs de pixels entre 0 et 1
image_flat = image_blur / 255.0

# Dimensions originales de l'image (hauteur, largeur)
original_shape = image_flat.shape

# Aplatir l'image : transforme l'image 2D (H x W) en un vecteur 1D (H*W,)
pixels = image_flat.flatten()

# Vérifications pour le débogage
print(" Forme de l'image après aplatissage :", pixels.shape)
print(" Dimensions originales :", original_shape)
print("\n")
print(" Valeurs min/max après normalisation :", pixels.min(), "/", pixels.max())

 Forme de l'image après aplatissage : (5992704,)
 Dimensions originales : (2448, 2448)


 Valeurs min/max après normalisation : 0.09411764705882353 / 0.996078431372549


## 4. Fuzzy C-Means: membership matrix initialisation

In [ ]:
# --- Étape : Initialisation du FCM ---

# Paramètres FCM
k = 6               # Nombre de clusters 
m = 1.7             # Paramètre de flou 
n = len(pixels)     # Nombre total de pixels 

# Fonction pour initialiser la matrice d'appartenance U
def initialize_membership_matrix(n, k):
    membership_mat = []
    for _ in range(n):
        random_values = np.random.dirichlet(np.ones(k), size=1)[0]  # Probas qui somment à 1
        membership_mat.append(random_values)
    return np.array(membership_mat)

# Initialisation de la matrice d'appartenance
U = initialize_membership_matrix(n, k)

# Vérification de l'initialisation
print(" Matrice d'appartenance initialisée forme :", U.shape)
print(" Exemple de vecteur d'appartenance :", U[0])
print(" Somme des appartenances :", np.sum(U[0]))

 Matrice d'appartenance initialisée forme : (5992704, 6)
 Exemple de vecteur d'appartenance : [0.28187697 0.19963587 0.08850729 0.30782177 0.02067646 0.10148164]
 Somme des appartenances : 1.0


## 5. Fuzzy C-Means main loop (centroids <-> memberships)

$$
c_i=\frac{\sum_j u_{ij}^m x_j}{\sum_j u_{ij}^m},\qquad
u_{ij}=\frac{1}{\sum_{k}\left(\frac{\lVert x_j-c_i\rVert}{\lVert x_j-c_k\rVert}\right)^{\frac{2}{m-1}}}
$$

The loop stops when `||U_new - U|| < epsilon` or after `max_iter` iterations.

In [ ]:
import numpy as np

# --- Étape : Calcul des centroïdes ---
def calculate_centroids(U, pixels, k, m):
    # Calcul des puissances des membres
    U_m = U ** m
    # Calcul du numérateur et du dénominateur pour tous les clusters en une seule opération
    num = np.dot(U_m.T, pixels)
    den = np.sum(U_m, axis=0)
    # Calcul des centroïdes en une seule opération
    centroids = num / den
    return centroids

# --- Étape : Mise à jour de la matrice d'appartenance ---
def update_membership_matrix(U, centroids, pixels, k, m):
    # Calcul des distances entre chaque pixel et chaque centroïde
    distances = np.abs(pixels[:, None] - centroids[None, :])
    # Calcul des ratios de distance
    dist_ratio = distances ** (2 / (m - 1))
    # Calcul de l'inverse de la distance
    inv_distances = 1 / dist_ratio
    # Calcul de la matrice d'appartenance en une seule opération
    U_new = inv_distances / np.sum(inv_distances, axis=1, keepdims=True)
    return U_new

# --- Initialisation FCM ---
k = 6               # Nombre de clusters (peut être ajusté selon l'image)
m = 1.7             # Paramètre de flou (>1)
n = len(pixels)     # Nombre total de pixels (valeurs normalisées)

# Fonction pour initialiser la matrice d'appartenance U
def initialize_membership_matrix(n, k):
    return np.random.dirichlet(np.ones(k), size=n)

# Initialisation de la matrice d'appartenance
U = initialize_membership_matrix(n, k)

# --- Calcul des centroïdes initiaux ---
centroids = calculate_centroids(U, pixels, k, m)

# Vérification initiale
print("Matrice d'appartenance initialisée forme :", U.shape)
print("Exemple de vecteur d'appartenance (1er pixel) :", U[0])
print("Somme des appartenances :", np.sum(U[0]))
print("Centroïdes initiaux calculés :", centroids)

#Boucle principale de FCM 
max_iter = 300
epsilon = 1e-5  # Seuil de convergence

for iteration in range(max_iter):
    print(f"\nItération {iteration + 1}")

    # Étape 1 : Calcul des centroïdes à partir de U
    centroids_old = centroids.copy()
    centroids = calculate_centroids(U, pixels, k, m)

    # Étape 2 : Mise à jour de la matrice U
    U_new = update_membership_matrix(U, centroids, pixels, k, m)

    # Évaluation de la différence
    diff = np.linalg.norm(U_new - U)
    print("Différence entre matrices d'appartenance :", diff)

    # Vérification de la convergence
    if diff < epsilon:
        print("Convergence atteinte.")
        break

    # Mise à jour de U pour l'itération suivante
    U = U_new

# Affichage des centroïdes finaux après convergence
print("\nCentroïdes finaux après convergence :")
for i, center in enumerate(centroids):
    print(f"Cluster {i+1}: {center:.4f}")

Matrice d'appartenance initialisée forme : (5992704, 6)
Exemple de vecteur d'appartenance (1er pixel) : [0.03474593 0.38096068 0.011977   0.17638686 0.16270179 0.23322774]
Somme des appartenances : 1.0
Centroïdes initiaux calculés : [0.42387825 0.42389813 0.42391916 0.42394714 0.42391394 0.42392261]

Itération 1
[... 807 lines of iteration log omitted ...]
Différence entre matrices d'appartenance : 1.0182201173828604e-05

Itération 271
Différence entre matrices d'appartenance : 9.448971200378545e-06
Convergence atteinte.

Centroïdes finaux après convergence :
Cluster 1: 0.3029
Cluster 2: 0.3830
Cluster 3: 0.4926
Cluster 4: 0.7185
Cluster 5: 0.4290
Cluster 6: 0.6028


## 6. Hard segmentation (argmax over memberships)

In [ ]:
# 1. Affecter chaque pixel au cluster ayant l'appartenance la plus élevée
cluster_labels = np.argmax(U, axis=1)

# 2. Reshaper les labels des clusters pour les mettre à la taille de l'image d'origine
cluster_labels_reshaped = cluster_labels.reshape(image_gray.shape)

# 3. Visualiser les différentes zones (clusters)
plt.figure(figsize=(4, 4))
plt.imshow(cluster_labels_reshaped, cmap='tab10')  # Colormap distincte pour chaque cluster
plt.title("Image segmentée avec FCM ", fontsize=10)
plt.axis('off')
plt.tight_layout()
plt.show()

# 4. Afficher les centres des clusters finaux
print("\nCentres des clusters finaux :")
for i, center in enumerate(centroids):
    print(f" - Cluster {i+1} : {center:.4f}")


Centres des clusters finaux :
 - Cluster 1 : 0.3029
 - Cluster 2 : 0.3830
 - Cluster 3 : 0.4926
 - Cluster 4 : 0.7185
 - Cluster 5 : 0.4290
 - Cluster 6 : 0.6028


## 7. Visual comparison with the ground-truth mask

In [ ]:
# Création de la figure pour afficher côte à côte
plt.figure(figsize=(12, 6))

# 3. Affichage de l'image segmentée par FCM
plt.subplot(1, 2, 1)
plt.imshow(cluster_labels_reshaped, cmap='tab10')  # Colormap distincte pour chaque cluster
plt.title("Image segmentée par FCM", fontsize=14)
plt.axis('off')

# 4. Affichage du masque
plt.subplot(1, 2, 2)
plt.imshow(mask_rgb)  # Afficher le masque original
plt.title("Masque original", fontsize=14)
plt.axis('off')

# Affichage final
plt.tight_layout()
plt.show()

## 8. Evaluation: pixel accuracy after optimal label matching

Predicted cluster ids are arbitrary, so they are matched one-to-one with the ground-truth classes using the
**Hungarian algorithm** (`scipy.optimize.linear_sum_assignment`) before computing the accuracy.

> **Note**: the ground-truth mask contains only 4 classes. With `k > 4`, a one-to-one matching necessarily leaves
> some clusters without a counterpart, which lowers the accuracy by construction (see `docs/results.md`).

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score
from scipy.optimize import linear_sum_assignment
import cv2
import matplotlib.pyplot as plt

def calculate_accuracy(true_mask, pred_mask):
    """Calcule l'accuracy après alignement des labels"""
    # Matrice de confusion
    conf_mat = np.zeros((true_mask.max()+1, pred_mask.max()+1), dtype=int)
    for i in range(true_mask.shape[0]):
        for j in range(true_mask.shape[1]):
            conf_mat[true_mask[i,j], pred_mask[i,j]] += 1
    
    # Alignement optimal des labels
    row_ind, col_ind = linear_sum_assignment(-conf_mat)
    
    # Réassignation des labels
    aligned_pred = np.zeros_like(pred_mask)
    for true_label, pred_label in zip(row_ind, col_ind):
        aligned_pred[pred_mask == pred_label] = true_label
    
    # Calcul de l'accuracy
    return accuracy_score(true_mask.flatten(), aligned_pred.flatten())

# Chargement du masque
mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

# Redimensionnement si nécessaire
if mask.shape != cluster_labels_reshaped.shape:
    mask = cv2.resize(mask, (cluster_labels_reshaped.shape[1], cluster_labels_reshaped.shape[0]))

# Calcul de l'accuracy
accuracy = calculate_accuracy(mask, cluster_labels_reshaped)

# Affichage simple
print(f"ACCURACY: {accuracy * 100:.2f}%")

ACCURACY: 45.13%


## Results reported in the thesis

| k | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|
| Accuracy (%) | 49.28 | 51.61 | **52.55** | 49.24 | 45.13 |

The value for **k = 6 (45.13 %, converged after 271 iterations)** is the one produced by this notebook.
The other values were obtained the same way by changing `k`.